# Phase 2 — Read-only Drive inventory and immutable snapshot

**Goal:** identify the governed source corpus and create a hash-verified snapshot inside the clean project.

This notebook performs **no OCR, no indexing, no LLM calls, and no source mutations**. It is valid only after the final verification cell prints `PHASE 2: PASS`.

Run with **Runtime → Run all**. Do not unzip the package manually.

## 1. Setup and explicit parameters

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile

EXPECTED_PACKAGE_SHA256 = "2036e445ca0bfca386071d454a6a494fcf2931f0ed1530e95b06b31589d5ca87"
PROJECT_FOLDER_NAME = "Devoteam_AI_CLEAN_PIPELINE"
PACKAGE_FILENAME = "PHASE_2_READ_ONLY_SNAPSHOT_PACKAGE.zip"

# Leave empty unless the notebook reports multiple master-workbook candidates.
MASTER_WORKBOOK_FILE_ID_OVERRIDE = ""

print("Phase 2 parameters loaded.")

In [ ]:
%pip -q install "google-api-python-client>=2.130,<3" "google-auth>=2.29,<3" "openpyxl>=3.1,<4" "PyYAML>=6,<7" "pytest>=8,<9"

## 2. Mount Drive and authenticate read access

In [ ]:
from google.colab import auth, drive
import google.auth
from googleapiclient.discovery import build

drive.mount("/content/drive", force_remount=False)
auth.authenticate_user()
credentials, _ = google.auth.default()
drive_service = build("drive", "v3", credentials=credentials, cache_discovery=False)
print("Drive mounted and API client authenticated.")

## 3. Locate the clean project and install the signed Phase 2 extension

In [ ]:
my_drive = Path("/content/drive/MyDrive")
candidates = [p.parent.parent for p in my_drive.rglob("config/project.yaml") if p.parent.parent.name == PROJECT_FOLDER_NAME]
candidates = sorted(set(path.resolve() for path in candidates))
assert len(candidates) == 1, f"Expected one {PROJECT_FOLDER_NAME} project, found: {candidates}"
PROJECT_ROOT = candidates[0]
PACKAGE_PATH = PROJECT_ROOT / PACKAGE_FILENAME
assert PACKAGE_PATH.exists(), f"Missing package: {PACKAGE_PATH}"

def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

actual_package_sha = file_sha256(PACKAGE_PATH)
assert actual_package_sha == EXPECTED_PACKAGE_SHA256, (
    f"Package hash mismatch. Expected {EXPECTED_PACKAGE_SHA256}, got {actual_package_sha}"
)

with zipfile.ZipFile(PACKAGE_PATH) as archive:
    members = archive.infolist()
    for member in members:
        target = (PROJECT_ROOT / member.filename).resolve()
        assert str(target).startswith(str(PROJECT_ROOT.resolve()) + os.sep), f"Unsafe archive path: {member.filename}"
    package_manifest = json.loads(archive.read("PHASE_2_PACKAGE_MANIFEST.json"))
    for entry in package_manifest["files"]:
        payload = archive.read(entry["path"])
        assert hashlib.sha256(payload).hexdigest() == entry["sha256"], f"Internal hash mismatch: {entry['path']}"
    for member in members:
        if member.is_dir() or member.filename == "PHASE_2_PACKAGE_MANIFEST.json":
            continue
        destination = PROJECT_ROOT / member.filename
        if destination.exists():
            existing = file_sha256(destination)
            packaged = hashlib.sha256(archive.read(member.filename)).hexdigest()
            assert existing == packaged, f"Refusing to overwrite changed project file: {member.filename}"
        else:
            archive.extract(member, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"Verified package SHA-256: {actual_package_sha}")
print("Phase 2 extension installed safely.")

## 4. Run local safety and unit tests

In [ ]:
environment = os.environ.copy()
environment["PYTHONPATH"] = str(PROJECT_ROOT / "src")
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", str(PROJECT_ROOT / "tests")],
    cwd=PROJECT_ROOT,
    env=environment,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, "Tests failed; source inventory has not started."
print("All foundation and Phase 2 tests passed.")

## 5. Inventory source metadata and create the immutable snapshot

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from devoteam_reference_ai.phase2_pipeline import run_phase2_snapshot

summary = run_phase2_snapshot(
    drive_service=drive_service,
    project_root=PROJECT_ROOT,
    config_path=PROJECT_ROOT / "config" / "phase2_source.yaml",
    master_workbook_file_id_override=MASTER_WORKBOOK_FILE_ID_OVERRIDE,
    progress=print,
)
print(json.dumps(summary, indent=2, ensure_ascii=False, sort_keys=True))

## 6. Verify every snapshot hash and publish the Phase 2 gate

In [ ]:
from devoteam_reference_ai.phase2_pipeline import verify_snapshot

snapshot_root = Path(summary["snapshot_root"])
verified = verify_snapshot(snapshot_root)
assert verified["status"] == "PASS"
assert verified["source_mutation_calls"] == 0
assert verified["external_llm_calls"] == 0
assert verified["ocr_calls"] == 0
assert verified["blocked_field_values_persisted"] == 0

print("PHASE 2: PASS")
print(f"Snapshot: {snapshot_root}")
print(f"Inventory items: {verified['source_items_inventoried']}")
print(f"Evidence files downloaded: {verified['evidence_files_downloaded']}")
print(f"Unavailable targets: {verified['evidence_targets_unavailable']}")
print("Next: review the generated Phase 2 report before Phase 3 extraction.")